# Retrieval Evaluation Notebook (PFE)

Notebook d'audit et de test de la phase retrieval (keyword / vector / hybrid) sur les index locaux.

Objectif:
- verifier que la retrieval fonctionne reellement,
- comparer les modes,
- tester les filtres metadata supportes,
- controler l'absence de fuite PII evidente,
- produire un verdict demo/experimental.

## 1) Prerequisites

Ce notebook suppose que les etapes precedentes sont deja executees:

1. extraction
2. chunking
3. anonymization
4. indexing

Artefacts attendus:
- `data/indexes/medical_rag.sqlite`
- `data/indexes/qdrant/`

Si Qdrant local est locke par un autre process, les tests vector/hybrid peuvent echouer.

In [ ]:
from __future__ import annotations

import json
import re
import sys
from collections import defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for c in [current] + list(current.parents):
        if (c / 'scripts' / 'retrieval').exists() and (c / 'data').exists():
            return c
    raise RuntimeError('Impossible de localiser la racine du repo.')


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

INDEX_DIR = repo_root / 'data' / 'indexes'
SQLITE_PATH = INDEX_DIR / 'medical_rag.sqlite'
QDRANT_DIR = INDEX_DIR / 'qdrant'
COLLECTION = 'medical_chunks'

print('Repo root:', repo_root)
print('Index dir :', INDEX_DIR)
print('SQLite    :', SQLITE_PATH)
print('Qdrant dir:', QDRANT_DIR)


In [ ]:
from scripts.retrieval.search import SearchEngine
from scripts.retrieval.models import RetrievalFilters

# Verification de presence des artefacts
checks = {
    'sqlite_exists': SQLITE_PATH.exists(),
    'qdrant_dir_exists': QDRANT_DIR.exists(),
    'indexing_report_exists': (INDEX_DIR / 'indexing_report.json').exists(),
    'index_manifest_exists': (INDEX_DIR / 'index_manifest.json').exists(),
}
print(json.dumps(checks, indent=2, ensure_ascii=False))

if not checks['sqlite_exists']:
    raise FileNotFoundError(f'SQLite manquant: {SQLITE_PATH}')
if not checks['qdrant_dir_exists']:
    raise FileNotFoundError(f'Qdrant dir manquant: {QDRANT_DIR}')


In [ ]:
# Initialisation du moteur retrieval
engine = SearchEngine(
    sqlite_path=SQLITE_PATH,
    qdrant_dir=QDRANT_DIR,
    collection=COLLECTION,
)
print('SearchEngine initialisé.')


In [ ]:
def truncate(text: str | None, n: int = 220) -> str:
    t = (text or '').strip()
    return t if len(t) <= n else t[: n - 3] + '...'


def run_query(query: str, mode: str = 'hybrid', top_k: int = 5, filters: RetrievalFilters | None = None, expand_context: bool = False):
    filters = filters or RetrievalFilters()
    resp = engine.search(
        query=query,
        mode=mode,
        top_k=top_k,
        filters=filters,
        expand_context=expand_context,
    )
    return resp


def response_to_table(resp, *, query: str, mode: str, top_k: int):
    rows = []
    for i, r in enumerate(resp.top_results[:top_k], start=1):
        m = r.metadata or {}
        rows.append({
            'query': query,
            'mode': mode,
            'top_k': top_k,
            'rank': i,
            'chunk_id': r.chunk_id,
            'doc_id': r.doc_id,
            'chunk_type': r.chunk_type,
            'score_vector': r.score_vector,
            'score_keyword': r.score_keyword,
            'score_hybrid': r.score_hybrid,
            'analyte': m.get('analyte'),
            'value_raw': m.get('value_raw'),
            'unit': m.get('unit'),
            'reference_range': m.get('reference_range'),
            'previous_result': m.get('previous_result'),
            'page_number': r.page_number,
            'row_index': m.get('row_index'),
            'source_kind': m.get('source_kind'),
            'source_pdf': r.source_pdf,
            'excerpt': truncate(r.text_preview, 220),
        })
    return pd.DataFrame(rows)


def first_hit_rank(df: pd.DataFrame, keywords: list[str]) -> int | None:
    if df.empty:
        return None
    kws = [k.lower() for k in keywords]
    for _, row in df.iterrows():
        blob = ' '.join([
            str(row.get('analyte') or ''),
            str(row.get('excerpt') or ''),
            str(row.get('chunk_type') or ''),
        ]).lower()
        if any(k in blob for k in kws):
            return int(row['rank'])
    return None


## 2) Smoke tests (manual)

Ces tests verifient rapidement que la retrieval retourne des chunks lisibles.

In [ ]:
smoke_queries = [
    'trichuris trichiura',
    'albumine 5 g/l',
]

smoke_tables = []
for q in smoke_queries:
    resp = run_query(q, mode='hybrid', top_k=5, filters=RetrievalFilters(), expand_context=True)
    df = response_to_table(resp, query=q, mode='hybrid', top_k=5)
    smoke_tables.append(df)
    display(Markdown(f"### Query: `{q}`"))
    display(df)
    display(Markdown(f"Context chunks: `{len(resp.context_chunks)}`"))


## 3) Exact analyte queries (keyword vs vector vs hybrid)

In [ ]:
exact_queries = {
    'calcitonine': ['calcitonine'],
    'peptide c': ['peptide c'],
    'insuline': ['insuline'],
    'pro bnp': ['pro bnp', 'nt-probnp', 'nt probnp'],
    'tshus': ['tshus', 'tsh'],
    'ferritine': ['ferritine'],
    'vitamine d': ['vitamine d'],
    'trichuris': ['trichuris'],
    'lithium': ['lithium'],
    'crp': ['crp', 'proteine c reactive'],
}

modes = ['keyword', 'vector', 'hybrid']
exact_rows = []
exact_detail = []

for q, clues in exact_queries.items():
    for mode in modes:
        try:
            resp = run_query(q, mode=mode, top_k=5, filters=RetrievalFilters(), expand_context=False)
            df = response_to_table(resp, query=q, mode=mode, top_k=5)
            exact_detail.append(df)
            rank = first_hit_rank(df, clues)
            exact_rows.append({
                'query': q,
                'mode': mode,
                'hit@1': rank == 1,
                'hit@3': (rank is not None and rank <= 3),
                'hit@5': (rank is not None and rank <= 5),
                'first_hit_rank': rank,
                'top1_chunk_id': None if df.empty else df.iloc[0]['chunk_id'],
                'top1_doc_id': None if df.empty else df.iloc[0]['doc_id'],
                'top1_chunk_type': None if df.empty else df.iloc[0]['chunk_type'],
            })
        except Exception as exc:
            exact_rows.append({
                'query': q,
                'mode': mode,
                'error': repr(exc),
                'hit@1': False,
                'hit@3': False,
                'hit@5': False,
                'first_hit_rank': None,
            })

exact_df = pd.DataFrame(exact_rows)
display(exact_df)


## 4) Semantic queries (keyword vs vector vs hybrid)

Ces requetes servent a voir si vector/hybrid apportent un gain sur des formulations plus libres.

In [ ]:
semantic_queries = {
    'resultat hormonal avec valeur de reference': ['tsh', 't4 libre', 'calcitonine', 'insuline', 'peptide c'],
    'resultat parasitologie positif': ['trichuris', 'ankylostoma', 'oeufs'],
    'resultat avec valeur superieure a la reference': ['>', 'eleve', 'high', 'above'],
    'resultat avec ancienne valeur': ['resultat anterieur', 'ancienne valeur', 'anterieur'],
    'analyse thyroidienne': ['tsh', 't4 libre', 't3 libre', 'anti-tpo', 'anti-tg'],
    'resultat biologique sans unite': ['non specifiee', 'unit_missing', 'sans unite'],
}

semantic_rows = []
for q, clues in semantic_queries.items():
    for mode in modes:
        try:
            resp = run_query(q, mode=mode, top_k=5, filters=RetrievalFilters(), expand_context=False)
            df = response_to_table(resp, query=q, mode=mode, top_k=5)
            rank = first_hit_rank(df, clues)
            semantic_rows.append({
                'query': q,
                'mode': mode,
                'hit@1': rank == 1,
                'hit@3': (rank is not None and rank <= 3),
                'hit@5': (rank is not None and rank <= 5),
                'first_hit_rank': rank,
                'top1_chunk_type': None if df.empty else df.iloc[0]['chunk_type'],
                'top1_doc_id': None if df.empty else df.iloc[0]['doc_id'],
            })
        except Exception as exc:
            semantic_rows.append({
                'query': q,
                'mode': mode,
                'error': repr(exc),
                'hit@1': False,
                'hit@3': False,
                'hit@5': False,
                'first_hit_rank': None,
            })

semantic_df = pd.DataFrame(semantic_rows)
display(semantic_df)


## 5) Metadata filters

Filtres supportes par `RetrievalFilters`:
- `document_type`, `doc_id`, `patient_id`, `sample_id`, `request_date`, `sample_type`, `chunk_type`, `source_pdf`

Non supportes directement dans le filtre actuel:
- `analyte_norm`
- `section`
- `source_kind`

Ces derniers peuvent etre verifies a posteriori dans les metadata retournees.

In [ ]:
filter_cases = [
    ('doc_id=report_1', RetrievalFilters(doc_id='report_1')),
    ('chunk_type=lab_result', RetrievalFilters(chunk_type='lab_result')),
    ('document_type=biology_report', RetrievalFilters(document_type='biology_report')),
    ('sample_type=SANG', RetrievalFilters(sample_type='SANG')),
    ('source_pdf=docs/report (1).pdf', RetrievalFilters(source_pdf='docs/report (1).pdf')),
]

filter_rows = []
for label, f in filter_cases:
    for mode in modes:
        try:
            resp = run_query('calcitonine', mode=mode, top_k=5, filters=f, expand_context=False)
            df = response_to_table(resp, query='calcitonine', mode=mode, top_k=5)
            all_match = True
            if not df.empty:
                if label == 'doc_id=report_1':
                    all_match = bool((df['doc_id'] == 'report_1').all())
                elif label == 'chunk_type=lab_result':
                    all_match = bool((df['chunk_type'] == 'lab_result').all())
                elif label == 'document_type=biology_report':
                    # available in metadata only, use response object
                    all_match = all((r.document_type == 'biology_report') for r in resp.top_results)
                elif label == 'source_pdf=docs/report (1).pdf':
                    all_match = bool((df['source_pdf'] == 'docs/report (1).pdf').all())
            filter_rows.append({
                'filter': label,
                'mode': mode,
                'top_count': len(resp.top_results),
                'all_match_filter': all_match,
                'top1_doc_id': None if df.empty else df.iloc[0]['doc_id'],
                'top1_chunk_type': None if df.empty else df.iloc[0]['chunk_type'],
            })
        except Exception as exc:
            filter_rows.append({'filter': label, 'mode': mode, 'error': repr(exc)})

filter_df = pd.DataFrame(filter_rows)
display(filter_df)


## 6) Aggregate metrics (simple)

Métriques calculées:
- Hit@1
- Hit@3
- Hit@5

Sur deux lots:
- exact queries
- semantic queries

In [ ]:
def summarize_hits(df: pd.DataFrame, label: str) -> pd.DataFrame:
    out = []
    for mode, g in df.groupby('mode'):
        ok = g[~g.get('error', pd.Series([None]*len(g))).notna()] if 'error' in g.columns else g
        n = len(ok)
        out.append({
            'suite': label,
            'mode': mode,
            'queries': n,
            'hit@1': float(ok['hit@1'].mean()) if n else 0.0,
            'hit@3': float(ok['hit@3'].mean()) if n else 0.0,
            'hit@5': float(ok['hit@5'].mean()) if n else 0.0,
        })
    return pd.DataFrame(out)

summary_df = pd.concat([
    summarize_hits(exact_df, 'exact'),
    summarize_hits(semantic_df, 'semantic'),
], ignore_index=True)

display(summary_df)


## 7) Anti-PII sanity check on retrieval outputs

Controle basique sur les top results retournes pendant les tests.

Remarque: les dates d'observation (`YYYY-MM-DD`) peuvent rester presentes car ce sont des dates cliniques utiles, pas forcement une fuite identitaire brute.

In [ ]:
forbidden_text_patterns = [
    'PATIENT TEST1',
    'PYXIS TEST',
    'patient_id_raw',
    'ip_patient',
    'patient_birth_date_raw',
    'prescripteur',
    'validé(e) par',
    'edité(e) par',
    'imprimé par',
    'anonymization_mapping',
    'data/private/',
]

forbidden_metadata_keys = {
    'patient_id', 'patient_id_raw', 'ip_patient',
    'patient_birth_date', 'patient_birth_date_raw',
    'prescriber', 'validated_by', 'edited_by', 'printed_by',
    'phone', 'fax'
}

pii_rows = []

def scan_df(df: pd.DataFrame, suite: str):
    if df.empty:
        return
    for _, row in df.iterrows():
        txt = str(row.get('excerpt') or '')
        hits = [p for p in forbidden_text_patterns if p.lower() in txt.lower()]
        pii_rows.append({
            'suite': suite,
            'query': row.get('query'),
            'mode': row.get('mode'),
            'rank': row.get('rank'),
            'forbidden_text_hits': hits,
        })

for df in exact_detail:
    scan_df(df, 'exact_detail')
for df in smoke_tables:
    scan_df(df, 'smoke')

pii_df = pd.DataFrame(pii_rows)
if pii_df.empty:
    display(Markdown('Aucun resultat a scanner.'))
else:
    pii_df['has_hits'] = pii_df['forbidden_text_hits'].apply(lambda x: bool(x))
    display(pii_df[pii_df['has_hits'] == True])

# Metadata keys check from top results of smoke + exact first rows
meta_key_violations = []
for df in exact_detail + smoke_tables:
    if df.empty:
        continue
    # metadata details are not all flattened in df; this check is done via live calls

# No assert hard-fail: this notebook is evaluative and must remain inspectable.
print('PII sanity check terminé (revue visuelle recommandée).')


## 8) Optional: run retrieval validation script

Ce script applique les cas de test dans `tests/retrieval_test_cases.json` et calcule des métriques globales (Recall@K, MRR, etc.).

In [ ]:
import subprocess

cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'retrieval' / 'validate_retrieval.py'),
    '--mode', 'all',
    '--top-k', '5',
    '--index-dir', str(INDEX_DIR),
    '--collection', COLLECTION,
    '--report', str(repo_root / 'data' / 'retrieval' / 'retrieval_validation_report.json'),
]

print('Commande:', ' '.join(cmd))
proc = subprocess.run(cmd, cwd=str(repo_root), capture_output=True, text=True)
print('exit_code =', proc.returncode)
print('--- stdout ---')
print(proc.stdout[:4000])
print('--- stderr ---')
print(proc.stderr[:2000])


## 9) Conclusion / Verdict

Interprétation recommandée:
- Si `hybrid` améliore Hit@3/Hit@5 vs `keyword` et `vector`, c'est un signal positif.
- Si les filtres metadata matchent correctement, c'est un bon signal de robustesse clinique.
- Si des fuites PII sont detectees, corriger avant toute demo publique.

Decision:
- Demo PFE: OK si smoke + exact queries sont coherentes et sans fuite evidente.
- Validation experimentale: OK si metrics stables sur `validate_retrieval.py`.
- Production-ready: non sans monitoring, jeu de tests elargi, et audit securite continu.

In [ ]:
# Fermer proprement
engine.close()
print('SearchEngine fermé.')
